# M3L4 E10 — LangGraph multiagente supervisor + Langfuse
### Módulo 3 · Lecture 4 · Construcción, pruebas y trazabilidad de agentes en producción

**Objetivo:** implementar el patrón supervisor en LangGraph — un nodo central que decide qué agente actúa, con historial de agentes visitados para evitar loops.

## Arquitectura supervisor
```
START
  ↓
supervisor_node  ← routing + control de loops
  ├── hr_agent_node
  ├── it_agent_node
  ├── finance_agent_node
  ├── legal_agent_node
  └── END (done=True)
```

## ¿Por qué supervisor?
El patrón supervisor permite:
- Controlar cuántos agentes se invocan
- Rastrear qué agentes ya actuaron (`visited_agents`)
- Detectar loops antes de que ocurran
- Centralizar la lógica de orquestación

In [ ]:
!pip install -q langfuse langchain langchain-openai langgraph
print('Instalación completa.')

In [ ]:
import os
from getpass import getpass

os.environ['LANGFUSE_PUBLIC_KEY'] = getpass('Langfuse Public Key: ')
os.environ['LANGFUSE_SECRET_KEY'] = getpass('Langfuse Secret Key: ')
os.environ['LANGFUSE_BASE_URL']   = 'https://cloud.langfuse.com'
os.environ['OPENAI_API_KEY']      = getpass('OpenAI API Key: ')
print('OK.')

In [ ]:
from typing import List
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END
from langfuse.langchain import CallbackHandler
print('Imports OK.')

In [ ]:
def route_query_v2(query):
    q = query.lower()
    hr_kw      = ['vacaciones','licencia','recibo','nómina','rrhh']
    it_kw      = ['vpn','error','app','laptop','wifi','login','contraseña']
    finance_kw = ['factura','pago','reembolso','gasto','cobro','comprobante','salario']
    legal_kw   = ['contrato','legal','confidencialidad','nda','acuerdo']
    det = []
    if any(w in q for w in hr_kw): det.append('hr')
    if any(w in q for w in it_kw): det.append('it')
    if any(w in q for w in finance_kw): det.append('finance')
    if any(w in q for w in legal_kw): det.append('legal')
    if len(det) > 1: return 'multi_intent'
    if len(det) == 1: return det[0]
    if len(q.split()) <= 2: return 'clarification'
    return 'general'

print('Router listo.')

## Estado con historial

In [ ]:
class SupervisorState(TypedDict):
    query: str
    intent: str
    visited_agents: List[str]  # agentes que ya actuaron en esta sesión
    response: str
    done: bool

print('SupervisorState definido.')

## TODO — Supervisor y agentes

Implementá el nodo supervisor que:
1. Detecta el intent
2. Verifica si el agente destino ya fue visitado (evitar loop)
3. Marca `done=True` si debe terminar

In [ ]:
def supervisor_node(state: SupervisorState) -> dict:
    """
    Nodo supervisor:
    - Detecta el intent
    - Si el intent es 'general'/'clarification' o el agente ya fue visitado, marca done=True
    - Retorna {'intent': intent, 'visited_agents': visited, 'done': bool}
    """
    # TODO
    pass

def hr_agent_node(state: SupervisorState) -> dict:
    """
    Agrega 'HRAgent' a visited_agents y retorna respuesta.
    done=True siempre (el agente cierra el turno)
    """
    # TODO
    pass

def it_agent_node(state: SupervisorState) -> dict:
    # TODO
    pass

def finance_agent_node(state: SupervisorState) -> dict:
    # TODO
    pass

def legal_agent_node(state: SupervisorState) -> dict:
    # TODO
    pass

print('Nodos definidos.')

## TODO — Función de routing condicional del supervisor

In [ ]:
def supervisor_route(state: SupervisorState) -> str:
    """
    Si done=True → 'end'
    Sino, mapea intent → nombre del nodo agente
    """
    # TODO
    pass

print('Función condicional definida.')

## TODO — Compilar el grafo

In [ ]:
# TODO: construir el grafo supervisor
# Estructura:
# START → supervisor_node
# supervisor_node → conditional_edges con supervisor_route
#   (mapear 'hr_agent', 'it_agent', 'finance_agent', 'legal_agent', 'end' → nodos)
# Cada agente_node → supervisor_node (vuelven al supervisor)
# Compilar

graph = None  # reemplazar
print('Grafo compilado.')

In [ ]:
if graph:
    print(graph.get_graph().draw_mermaid())

In [ ]:
queries = [
    '¿Cómo solicito mis días de vacaciones?',
    'Mi VPN no conecta desde ayer',
    'Necesito ver mi factura del mes pasado',
    'ayuda'
]

if graph:
    for q in queries:
        lf = CallbackHandler()
        output = graph.invoke(
            {'query': q, 'intent': '', 'visited_agents': [], 'response': '', 'done': False},
            config={'callbacks': [lf], 'metadata': {'langfuse_tags': ['m3l4','supervisor']}}
        )
        print(f'Query:   {q[:45]}')
        print(f'Intent:  {output["intent"]}')
        print(f'Visited: {output["visited_agents"]}')
        print(f'Resp:    {output["response"][:60]}...')
        print()

In [ ]:
if graph:
    lf = CallbackHandler()
    r = graph.invoke({'query': 'No puedo ver mi factura', 'intent': '', 'visited_agents': [], 'response': '', 'done': False},
                     config={'callbacks': [lf]})
    assert r['intent'] == 'finance'
    assert 'FinanceAgent' in r['visited_agents']
    assert r['done'] == True
    assert len(r['response']) > 5
    print('Checks E10 OK ✅')